In [ ]:
import sys
sys.path.append("../")

import folium
from folium.plugins import TimestampedGeoJson
import pandas as pd
import geopandas as gpd

import src.paths as PATHS
import src.constants as CONST

In [ ]:
sam_data = PATHS.DATA_DIR / "sam" / "sam_processed.gpkg"

sam_dict = {}
for _, layer_data in gpd.list_layers(sam_data).iterrows():
    sam_dict[layer_data["name"]] = gpd.read_file(sam_data, layer=layer_data["name"])

In [ ]:
sam_dict.keys()

In [ ]:
sam_dict["Ingetekende _Erosie_Rijntakken_2024"].head()

In [ ]:
sam_dict["20250915_Pt"]["observation_date"] = pd.to_datetime(sam_dict["20250915_Pt"]["observation_date"])

sam_dict["20250915_Pt"].sample(5)

In [ ]:
luke_data = PATHS.DATA_DIR / "phase1_2025-08-14_v1.gpkg"

luke_dict = {}
for _, layer_data in gpd.list_layers(luke_data).iterrows():
    luke_dict[layer_data["name"]] = gpd.read_file(luke_data, layer=layer_data["name"])

In [ ]:
luke_dict.keys()

In [ ]:
luke_dict["punten_oever"].sample(5)

In [ ]:
luke_dict["punten_oever"]["observation_date"] = pd.to_datetime(luke_dict["punten_oever"]["dtm_date"].map(lambda x: f"{x}-06-01"))

In [ ]:
mapa = folium.Map(location=[CONST.CENTRE_NL_LAT, CONST.CENTRE_NL_LON], zoom_start=CONST.DEFAULT_NL_ZOOM, control_scale=True)

# add scope
fg_scope = folium.FeatureGroup(name="scope regions", show=True).add_to(mapa)
folium.GeoJson(luke_dict["vlakken_scope"]).add_to(fg_scope)

# add SAM data
sam_layer = folium.FeatureGroup(name="SAM detections", show=True).add_to(mapa)

sam_dict["20250915_Pt"]["origin"] = "SAM"
luke_dict["punten_oever"]["origin"] = "HM"

all_points = pd.concat([sam_dict["20250915_Pt"].to_crs(epsg=CONST.EPSG_WGS84), luke_dict["punten_oever"].to_crs(epsg=CONST.EPSG_WGS84)], ignore_index=True)
all_points = all_points.loc[all_points.within(luke_dict["vlakken_scope"].to_crs(epsg=CONST.EPSG_WGS84)["geometry"].union_all())]

geojson_features = []
for _, row in all_points.iterrows():
    # TODO: lines are not rendered even though this looks exactly like the example in the docs where lines are rendered?!
    point = row["geometry"].__geo_interface__
    # line["coordinates"] = [list(a) for a in line['coordinates']]  # properly deal with a LineString
    geojson_features.append({
        "type": "Feature",
        "geometry": point,
        "properties": { 
            "times": [row["observation_date"].strftime("%Y-%m-%dT%H:%M:%S")],
            "icon": "circle",
            "style": {"color": "orange" if row["origin"] == "SAM" else "blue", "fillOpacity": 0.6, "radius_m": 2, "fillColor": "orange" if row["origin"] == "SAM" else "blue"},
        },
    })

geojson_data = {"type": "FeatureCollection", "features": geojson_features}
TimestampedGeoJson(
    geojson_data,
    period="P1Y",
    duration="P11M",
    transition_time=200,  # Milliseconds between frames
    loop=False,            # Loop animation
    auto_play=False,      # Start playing automatically
    loop_button=True,
).add_to(mapa)


folium.LayerControl().add_to(mapa)

mapa.save("sam_vs_height.html")